# Preprocesamiento: fotogramas válidos y contexto temporal

Primera fase del preprocesamiento. Para cada imagen capturada:

1. Calcula métricas de calidad visual (resolución, brillo, nitidez) con `src/calidad.py`.
2. Añade el contexto temporal con `src/tiempo.py`:
   - `timestamp` (UTC, del nombre del fichero) y `timestamp_local` (Madrid).
   - `time_slot_2h` y `day_type` (laborable / fin_de_semana / festivo) en hora local.
   - `illumination` (día / crepúsculo / noche) por elevación solar. Es independiente del clima, que etiqueta `02_preprocess_labelled_weather`.
3. Filtra imágenes no válidas y cámaras en lista negra.
4. Muestrea como máximo un fotograma por cámara cada X segundos, siendo X el refresco medido en `00b` (no tiene sentido muestrear más rápido de lo que la cámara publica imágenes nuevas).

**Salida:** `data/malaga/fases/frames.parquet` (fase `frames`) y `data/malaga/fases/blacklist_camaras.csv`. Las fases `weather` y `detections` se escriben en ficheros propios y se unen por `(camera, image)` con `pipeline.load_frames()`.

Umbrales y parámetros: `config.yaml` (`calidad`, `muestreo`, `iluminacion`).

## 1. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from calidad import add_quality, camera_blacklist, image_metrics, list_camera_images
from config import load_config, path, relative
from pipeline import sample_every, sampling_seconds, save_phase
from tiempo import day_type, illumination, load_holidays, parse_timestamp_from_name, time_slot_2h, to_local

CFG = load_config()
RAW_DIR = path("raw")
BLACKLIST_FILE = path("fases") / "blacklist_camaras.csv"

HOLIDAYS = load_holidays(path("festivos"))
LAT, LON = CFG["ubicacion"]["lat"], CFG["ubicacion"]["lon"]
TWILIGHT = CFG["iluminacion"]["elevacion_crepusculo"]

FRAME_SAMPLE_SECONDS, sampling_source = sampling_seconds()

print(f"Directorio de imágenes: {RAW_DIR}")
print(f"Festivos cargados: {len(HOLIDAYS)}")
print(f"Muestreo temporal: {FRAME_SAMPLE_SECONDS} s ({sampling_source})")

## 2. Métricas de calidad y contexto temporal por imagen

In [ ]:
image_files = list_camera_images(RAW_DIR)
print("Imágenes encontradas:", len(image_files))

rows = []
for img_path in image_files:
    try:
        metrics = image_metrics(img_path)
    except ValueError:
        continue

    ts = parse_timestamp_from_name(img_path.name)
    ts_local = to_local(ts)
    metrics.update({
        "path": relative(img_path),
        "timestamp": ts,
        "timestamp_local": ts_local,
        "time_slot_2h": time_slot_2h(ts_local),
        "day_type": day_type(ts_local, HOLIDAYS),
        "illumination": illumination(ts, LAT, LON, TWILIGHT),
    })
    rows.append(metrics)

image_quality_df = pd.DataFrame(rows)
print("Imágenes analizadas:", len(image_quality_df))
print("Sin marca de tiempo:", image_quality_df["timestamp"].isna().sum())
display(image_quality_df.head())

## 3. Criterios de calidad y lista negra

- El brillo mínimo solo se exige a imágenes de día.
- El umbral de nitidez es un percentil de la distribución observada.
- Una cámara entra en la lista negra solo si falla más del `max_fraccion_fallos` de sus imágenes de día; una imagen aislada mala ya no descarta la cámara.

La lista negra se recalcula en cada ejecución (no se acumula), de modo que una cámara que mejora vuelve a entrar.

In [ ]:
image_quality_df, thresholds = add_quality(image_quality_df, CFG["calidad"])
print("Umbrales:", thresholds)
print("\nImágenes válidas por iluminación:")
display(image_quality_df.groupby("illumination")["valid"].agg(["size", "mean"]))

camera_stats = camera_blacklist(image_quality_df, CFG["calidad"]["max_fraccion_fallos"])
BLACKLIST_FILE.parent.mkdir(parents=True, exist_ok=True)
camera_stats.to_csv(BLACKLIST_FILE, index=False)

blacklisted = set(camera_stats.loc[camera_stats["en_lista_negra"], "camera"])
print(f"\nCámaras en lista negra: {len(blacklisted)} de {len(camera_stats)} -> {BLACKLIST_FILE}")
display(camera_stats.sort_values("fraccion_fallos", ascending=False).head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for illum, group in image_quality_df.groupby("illumination"):
    axes[0].hist(group["brightness"], bins=40, alpha=0.5, label=illum)
axes[0].axvline(thresholds["brightness_min_dia"], color="k", ls="--", label="mín. día")
axes[0].set_title("Brillo por iluminación")
axes[0].legend()
axes[1].hist(image_quality_df["sharpness"], bins=60, log=True)
axes[1].axvline(thresholds["sharpness_min"], color="k", ls="--", label="umbral (percentil)")
axes[1].set_title("Nitidez (varianza del Laplaciano)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Fotogramas válidos y muestreo temporal

In [ ]:
valid = image_quality_df[image_quality_df["valid"] & ~image_quality_df["camera"].isin(blacklisted)]
frames = sample_every(valid, FRAME_SAMPLE_SECONDS)
frames["frame_interval_seconds"] = FRAME_SAMPLE_SECONDS

print("Fotogramas válidos antes de muestreo:", len(valid))
print("Fotogramas tras muestreo temporal:", len(frames))
for col in ["day_type", "time_slot_2h", "illumination"]:
    print(f"\n{col}:\n{frames[col].value_counts(dropna=False).sort_index().to_string()}")

## 5. Exportar la fase `frames`

In [ ]:
export_cols = [
    "camera",
    "image",
    "path",
    "timestamp",
    "timestamp_local",
    "time_slot_2h",
    "day_type",
    "illumination",
    "width",
    "height",
    "brightness",
    "sharpness",
    "quality_score",
    "frame_interval_seconds",
]
out = save_phase(frames[export_cols], "frames")
print(f"Fase frames: {len(frames)} fotogramas -> {out}")
display(frames[export_cols].head(10))